<img src="https://gradientflow.com/wp-content/uploads/2024/01/newsletter94-function-calling.jpeg" width=800>



---
# Creating LLM Functions

## OpenAI Function Schema

In [ ]:
pip install python-dotenv

In [ ]:
pip install openai==0.28

In [ ]:
pip install python-dotenv

In [ ]:
from dotenv import load_dotenv
import os
import openai

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
tools = [
    # 1. Get Exchange Rate
    {
        "type": "function",
        "function": {
            "name": "get_exchange_rate",
            "description": "Get the current exchange rate of a base currency and target currency",
            "parameters": {
                "type": "object",
                "properties": {
                    "base_currency": {
                        "type": "string",
                        "description": "The base currency code, e.g. USD, EUR, JPY"
                    },
                    "target_currency": {
                        "type": "string",
                        "description": "The target currency code, e.g. USD, EUR, JPY"
                    },
                    "date": {
                        "type": "string",
                        "description": "Optional: specific date (YYYY-MM-DD) for historical rates"
                    }
                },
                "required": ["base_currency", "target_currency"]
            }
        }
    },
    # 2. Search Internet
    {
        "type": "function",
        "function": {
            "name": "search_internet",
            "description": "Retrieve real-time search results from the web",
            "parameters": {
                "type": "object",
                "properties": {
                    "search_query": {
                        "type": "string",
                        "description": "The query to search for"
                    }
                },
                "required": ["search_query"]
            }
        }
    },
    # 3. Send Email
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": "Send an email to one or more recipients",
            "parameters": {
                "type": "object",
                "properties": {
                    "to": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of recipient email addresses"
                    },
                    "subject": {
                        "type": "string",
                        "description": "Subject line of the email"
                    },
                    "body": {
                        "type": "string",
                        "description": "Body text of the email"
                    }
                },
                "required": ["to", "subject", "body"]
            }
        }
    },
    # 4. Send Message
    {
        "type": "function",
        "function": {
            "name": "send_message",
            "description": "Send an instant message to a contact",
            "parameters": {
                "type": "object",
                "properties": {
                    "to": {
                        "type": "string",
                        "description": "Recipient identifier (e.g. username or phone number)"
                    },
                    "message": {
                        "type": "string",
                        "description": "The message content"
                    }
                },
                "required": ["to", "message"]
            }
        }
    },
    # 5. Create Document
    {
        "type": "function",
        "function": {
            "name": "create_document",
            "description": "Create a new document with a given title and content",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string",
                        "description": "Title of the new document"
                    },
                    "content": {
                        "type": "string",
                        "description": "Initial content for the document"
                    }
                },
                "required": ["title", "content"]
            }
        }
    },
    # 6. Edit Document
    {
        "type": "function",
        "function": {
            "name": "edit_document",
            "description": "Edit an existing document by ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "document_id": {
                        "type": "string",
                        "description": "Identifier of the document to edit"
                    },
                    "changes": {
                        "type": "string",
                        "description": "Instructions or new content to apply"
                    }
                },
                "required": ["document_id", "changes"]
            }
        }
    },
    # 7. Fetch Weather
    {
        "type": "function",
        "function": {
            "name": "fetch_weather",
            "description": "Get the weather forecast for a given location and date",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City or coordinates, e.g. Karachi or 24.86,67.01"
                    },
                    "date": {
                        "type": "string",
                        "description": "Optional: forecast date (YYYY-MM-DD) or 'today', 'tomorrow'"
                    }
                },
                "required": ["location"]
            }
        }
    },
    # 8. Schedule Meeting
    {
        "type": "function",
        "function": {
            "name": "schedule_meeting",
            "description": "Schedule a calendar meeting with participants",
            "parameters": {
                "type": "object",
                "properties": {
                    "participants": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Emails or IDs of meeting attendees"
                    },
                    "date": {
                        "type": "string",
                        "description": "Meeting date in YYYY-MM-DD format"
                    },
                    "time": {
                        "type": "string",
                        "description": "Meeting start time in HH:MM (24-hour)"
                    },
                    "subject": {
                        "type": "string",
                        "description": "Title or subject of the meeting"
                    }
                },
                "required": ["participants", "date", "time"]
            }
        }
    }
]


In [ ]:
import json

def call_with_tools(prompt, model="gpt-4o-mini"):
    response = openai.ChatCompletion.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        tools=tools,            # your function definitions
        tool_choice="auto"      # allow function-calling
    )
    msg = response.choices[0].message

    # Did the model pick a tool?
    if msg.get("tool_calls"):
        call = msg.tool_calls[0]
        name = call.function.name
        args = json.loads(call.function.arguments)
        print(f"🛠️  Model called: {name} with {args}")
    else:
        print("✏️  Model replied without calling a tool:")
        print(msg.content)

    return msg


In [ ]:
tests = [
    ("What’s the USD to EUR rate today?",                                   "get_exchange_rate"),
    ("Search for the latest news on electric vehicles.",                     "search_internet"),
    ("Send a quick note to Alice about today’s meeting.",                    "send_message"),
    ("Email Bob the agenda for tomorrow’s call.",                            "send_email"),
    ("Create a document summarizing our Q2 revenue.",                       "create_document"),
    ("Edit the financial report to update last month’s numbers.",            "edit_document"),
    ("What’s the weather forecast for Karachi this weekend?",                "fetch_weather"),
    ("Set up a meeting with the product team next Wednesday at 2 PM.",      "schedule_meeting"),
]


In [ ]:
results = []

for prompt, expected in tests:
    msg = call_with_tools(prompt)
    actual = msg.tool_calls[0].function.name if msg.get("tool_calls") else "<none>"
    ok = (actual == expected)
    results.append((prompt, expected, actual, ok))
    status = "✅" if ok else "❌"
    print(f"{status} “{prompt[:40]}…” ➞ got `{actual}`, expected `{expected}`")

# Summary
fails = [r for r in results if not r[3]]
print(f"\nTested {len(results)} prompts: {len(fails)} mis‐routed, {len(results)-len(fails)} correct.")
if fails:
    print("Mismatches:")
    for prompt, expected, actual, _ in fails:
        print(f" • “{prompt}”  expected `{expected}`, got `{actual}`")
